# Fine-tuning for a new language pair

In this activity, you will fine-tune the pre-trained English-French attention model for English-German translation. This demonstrates **transfer learning** in neural machine translation.

## Key concepts

**Transfer learning** leverages knowledge from one task to improve performance on a related task. For NMT, what actually transfers between language pairs:

**What transfers effectively:**
- **Architectural capacity:** The LSTM has learned how to maintain long-term dependencies and what information to store in hidden states
- **Attention mechanism patterns:** The model has learned how to selectively attend to relevant input positions when generating output
- **Sequence generation skills:** The decoder has learned autoregressive generation strategies and fluency patterns
- **Optimization landscape:** Pre-trained weights start in a better region of parameter space than random initialization

**What doesn't transfer (language-specific):**
- **Embedding layers:** Vocabulary mappings are specific to each language
- **Word order patterns:** While the encoder is tuned to the source language's syntax, some structural knowledge helps convergence
- **Morphological patterns:** Language-specific grammatical structures

**Why English-French → English-German works well:**
- Both pairs share the same source language (English)
- French and German are typologically similar (both European languages, mostly SVO word order, similar morphology)
- The real benefit: fewer epochs needed because we're not learning attention and generation mechanisms from scratch

**Important caveat:** Transfer learning is most effective between linguistically similar language pairs. Transfer from English-French to English-Japanese or English-Arabic would be less effective due to vastly different word orders, scripts, and grammatical structures.

**Benefits of fine-tuning:**
- Faster convergence (5 epochs vs. 15+ epochs from scratch)
- Better final performance with less data
- Reuses learned attention and generation patterns

## Your task

1. Download the pre-trained English-French attention model from Hugging Face Hub
2. Replace embedding layers with new English-German vocabulary
3. Fine-tune on English-German data (OPUS-100 corpus)
4. Compare training speed and final BLEU score with training from scratch

### References

Transfer learning in NMT:

> Zoph, B., Yuret, D., May, J., & Knight, K. (2016). **Transfer learning for low-resource neural machine translation.** *Proceedings of the 2016 Conference on Empirical Methods in Natural Language Processing (EMNLP).* https://arxiv.org/abs/1604.02201

## Notebook set-up

### Imports

In [ ]:
import logging
import os

# Environment variables for TensorFlow. Note: these must
# be set BEFORE importing TensorFlow to take effect.
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' # Suppress TensorFlow warnings
os.environ['CUDA_VISIBLE_DEVICES'] = '1' # Select GPU, 0 for GPU 1, etc.

# Core libraries
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# NLP and translation libraries
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from sacrebleu.metrics import BLEU
from transformers import MarianTokenizer

# Keras model components
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Dense, Embedding, Bidirectional, Concatenate, Attention
)

### Configuration

In [ ]:
# Configure GPU memory growth to avoid OOM errors
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# Disable Jupyter widgets to prevent rendering issues after reopening notebook
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '0'  # Keep progress bars but use plain text

# Set event level output filter for TensorFlow
logging.getLogger('tensorflow').setLevel(logging.ERROR)

# Initialization
np.random.seed(315)
tf.random.set_seed(315)

## 1. Prepare English-German data

Load the OPUS-100 English-German parallel corpus. We'll use the same data preparation pipeline as the English-French demo.

### 1.1. Load tokenizer

In [ ]:
# Load pre-trained subword tokenizer for English-German
# TODO: Load the appropriate MarianTokenizer for English-German translation
# Hint: Use 'Helsinki-NLP/opus-mt-en-de' instead of 'opus-mt-en-fr'
tokenizer = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-en-de')

print(f'Tokenizer vocabulary size: {tokenizer.vocab_size}')
print(f'Special tokens: {tokenizer.special_tokens_map}')

# Example of subword tokenization
example = 'The neural network learned representations'
tokens = tokenizer.tokenize(example)
print(f'\nExample tokenization:')
print(f'  Input: "{example}"')
print(f'  Tokens: {tokens}')

### 1.2. Load dataset

In [ ]:
# Load OPUS-100 English-German translation dataset
# TODO: Load the 'en-de' language pair instead of 'en-fr'
dataset = load_dataset('Helsinki-NLP/opus-100', 'en-de')

# Extract translation pairs and filter by token length
pairs = []
MAX_SEQ_LENGTH = 20  # Maximum tokens per sentence

for item in dataset['train']:

    en_text = item['translation']['en'].strip()
    de_text = item['translation']['de'].strip()
    
    # Check token length using tokenize() to avoid truncation warnings
    en_tokens = tokenizer.tokenize(en_text)
    de_tokens = tokenizer.tokenize(de_text)
    
    if len(en_tokens) <= MAX_SEQ_LENGTH and len(de_tokens) <= MAX_SEQ_LENGTH:
        pairs.append((en_text, de_text))
    
    # Limit dataset size for reasonable training time
    if len(pairs) >= 100000:
        break

print(f'Loaded {len(pairs)} translation pairs')
print(f'\nSample pairs:')

for en, de in pairs[1:5]:
    print()
    print(f'  EN: {en}')
    print(f'  DE: {de}')

In [ ]:
# Tokenize all pairs using the subword tokenizer
MAX_ENCODER_LEN = 22  # Slightly larger than MAX_SEQ_LENGTH for special tokens
MAX_DECODER_LEN = 24

# Tokenize source (English) sentences
encoder_inputs = tokenizer(
    [pair[0] for pair in pairs],
    padding='max_length',
    truncation=True,
    max_length=MAX_ENCODER_LEN,
    return_tensors='np'
)

# Tokenize target (German) sentences
decoder_input_texts = [pair[1] for pair in pairs]

decoder_inputs = tokenizer(
    decoder_input_texts,
    padding='max_length',
    truncation=True,
    max_length=MAX_DECODER_LEN - 1,  # Leave room for BOS token
    return_tensors='np'
)

# Prepare encoder data
encoder_input_data = encoder_inputs['input_ids']

# CRITICAL: Decoder input must start with BOS token (we use pad_token_id)
# decoder_input: [BOS, tok1, tok2, ..., tokN, pad, pad...]
# decoder_target: [tok1, tok2, ..., tokN, EOS, pad, pad...]
raw_decoder_tokens = decoder_inputs['input_ids']
decoder_input_data = np.full((len(pairs), MAX_DECODER_LEN), tokenizer.pad_token_id, dtype=np.int32)
decoder_input_data[:, 1:1 + raw_decoder_tokens.shape[1]] = raw_decoder_tokens

# Targets are the original tokens (what we want to predict after BOS)
decoder_target_data = np.full((len(pairs), MAX_DECODER_LEN), tokenizer.pad_token_id, dtype=np.int32)
decoder_target_data[:, :raw_decoder_tokens.shape[1]] = raw_decoder_tokens

# Model dimensions
num_samples = len(pairs)
num_tokens = tokenizer.vocab_size  # Same vocab for encoder and decoder
max_encoder_len = MAX_ENCODER_LEN
max_decoder_len = MAX_DECODER_LEN

print(f'Vocabulary size: {num_tokens}')
print(f'Max encoder length: {max_encoder_len}')
print(f'Max decoder length: {max_decoder_len}')
print(f'Training samples: {num_samples}')
print(f'\nEncoder input shape: {encoder_input_data.shape}')
print(f'Decoder input shape: {decoder_input_data.shape}')
print(f'Decoder target shape: {decoder_target_data.shape}')

## 2. Load pre-trained model and prepare for fine-tuning

### 2.1. Build base model with English-German vocabulary

First, we build a model with the new vocabulary size. This gives us the correct architecture.

In [ ]:
# Import model building function from src module
import sys
sys.path.append('..')

from src import build_attention_model

# Build the model with English-German vocabulary
model = build_attention_model(num_tokens, max_encoder_len, max_decoder_len, latent_dim=256)
model.summary()

### 2.2. Download and load pre-trained weights (except embeddings)

Now we download the pre-trained English-French model from Hugging Face Hub and selectively load its weights, excluding the embedding layers since they're language-specific.

**Key insight:** The LSTM encoder/decoder and attention mechanism have learned valuable skills for sequence-to-sequence modeling: how to maintain context in hidden states, when to attend to different input positions, and how to generate fluent sequences. These capabilities transfer reasonably well between typologically similar language pairs like French and German (both European languages with similar syntax).

**What we reuse vs. reinitialize:**
- ✓ **Reuse:** LSTM parameters (gate weights, hidden state transformations), attention mechanism (learned attention patterns)
- ✗ **Reinitialize:** Embedding layers (vocabulary is language-specific - German words need different embeddings than French words)

In [ ]:
from huggingface_hub import hf_hub_download

# Download pre-trained English-French attention model weights from Hugging Face Hub
# TODO: Download the model weights using hf_hub_download()
# Hint: repo_id = 'gperdrizet/english-french-LSTM-attention', filename = 'model_weights.h5'

print('Downloading pre-trained English-French model from Hugging Face Hub...')

try:
    pretrained_weights_path = hf_hub_download(
        repo_id='gperdrizet/english-french-LSTM-attention',
        filename='model_weights.h5',
        cache_dir='../models/cache'
    )
    
    print(f'Model downloaded successfully to: {pretrained_weights_path}')
    
    # Load all weights first
    model.load_weights(pretrained_weights_path)
    
    # Reinitialize embedding layers (language-specific)
    # These need to learn English-German mappings from scratch
    encoder_embedding = model.get_layer('encoder_embedding')
    decoder_embedding = model.get_layer('decoder_embedding')
    
    # Reinitialize embeddings with random weights
    encoder_embedding.set_weights([
        np.random.uniform(-0.05, 0.05, encoder_embedding.get_weights()[0].shape)
    ])
    decoder_embedding.set_weights([
        np.random.uniform(-0.05, 0.05, decoder_embedding.get_weights()[0].shape)
    ])
    
    print('\nTransfer learning setup complete!')
    print('✓ Reusing: LSTM encoder, LSTM decoder, attention mechanism')
    print('✗ Reinitializing: Encoder embeddings, decoder embeddings')
    print('\nThis should converge much faster than training from scratch!')

except Exception as e:
    print(f'ERROR: Could not download model from Hugging Face Hub: {e}')
    print('\nThe pre-trained model may not be available yet.')
    print('Proceeding with training from scratch instead...')

### 2.3. Set up callbacks

Configure BLEU evaluation, checkpointing, and TensorBoard logging.

In [ ]:
from src import BLEUCallback, build_inference_models_attention, translate_attention

# Create BLEU callback
bleu_callback = BLEUCallback(
    pairs=pairs,
    tokenizer=tokenizer,
    max_encoder_len=max_encoder_len,
    max_decoder_len=max_decoder_len,
    translate_fn=translate_attention,
    build_inference_fn=lambda model, latent_dim: build_inference_models_attention(model, max_encoder_len, latent_dim),
    sample_size=100,
    latent_dim=256,
    restore_best_weights=True
)

## 3. Fine-tune the model

**Important:** Fine-tuning typically requires fewer epochs (3-5) than training from scratch (15+). The pre-trained weights give us a head start.

In [ ]:
%%time

import os
from datetime import datetime
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard

# Create directories for checkpoints and logs
checkpoint_dir = '../models/checkpoints/lstm-attention-en-de'
log_dir = '../logs/lstm-attention-en-de'
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

# Model checkpoint callback
checkpoint_callback = ModelCheckpoint(
    filepath=os.path.join(checkpoint_dir, 'model_epoch_{epoch:02d}_val_loss_{val_loss:.4f}.h5'),
    monitor='val_loss',
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)

# TensorBoard callback
tensorboard_callback = TensorBoard(
    log_dir=os.path.join(log_dir, datetime.now().strftime('%Y%m%d-%H%M%S')),
    histogram_freq=1,
    write_graph=True,
    write_images=False,
    update_freq='epoch'
)

# TODO: Fine-tune the model
# Hint: Use fewer epochs (5-7) since we're fine-tuning, not training from scratch
# Try a slightly lower learning rate for fine-tuning

# Optional: Use a lower learning rate for fine-tuning
# tf.keras.backend.set_value(model.optimizer.learning_rate, 0.0005)

history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=32,
    epochs=5,  # Much fewer epochs needed for fine-tuning!
    validation_split=0.1,
    verbose=1,
    callbacks=[bleu_callback, checkpoint_callback, tensorboard_callback]
)

print(f'\nFinal training loss: {history.history["loss"][-1]:.4f}')
print(f'Final validation loss: {history.history["val_loss"][-1]:.4f}')
print(f'Best BLEU score: {bleu_callback.best_bleu:.2f}\n')

### 3.1. Learning curves

**Viewing TensorBoard logs:** To visualize training metrics (loss, accuracy, BLEU score) in TensorBoard, run:
```bash
tensorboard --logdir ../logs/lstm-attention-en-de
```
Then open the provided URL in your browser.

**Expected behavior:** With transfer learning, you should see:
- Faster initial convergence (loss drops quickly in first epoch)
- Higher starting BLEU score (model already knows attention patterns)
- Good final BLEU after just 5 epochs

In [ ]:
# Plot learning curves: loss, accuracy, and BLEU
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(10, 3))

# Epoch where best BLEU was achieved
best_epoch = bleu_callback.best_epoch

# Left plot: training vs validation loss
axes[0].set_title('Loss')
axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].axvline(x=best_epoch, color='red', linestyle=':', label='Best model')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend(loc='best')

# Middle plot: token-level accuracy
axes[1].set_title('Token accuracy')
axes[1].plot(history.history['accuracy'], label='Train')
axes[1].plot(history.history['val_accuracy'], label='Validation')
axes[1].axvline(x=best_epoch, color='red', linestyle=':', label='Best model')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend(loc='best')

# Right plot: BLEU score over training
axes[2].set_title('Validation BLEU score')
axes[2].plot(bleu_callback.bleu_scores, c='black', label='BLEU score')
axes[2].axvline(x=best_epoch, color='red', linestyle=':', label='Best model')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('BLEU')
axes[2].legend(loc='best')

plt.tight_layout()
plt.show()

## 4. Evaluate English-German translations

In [ ]:
# Build final inference models
encoder_model, decoder_model = build_inference_models_attention(model, max_encoder_len, latent_dim=256)

# Test translation on sample sentences
test_sentences = [
    'Hello, how are you?',
    'I love programming.',
    'The weather is nice today.',
    'Machine learning is fascinating.',
    'Thank you for your help.'
]

print('Sample translations (English → German):')

for sent in test_sentences:

    print(f'  EN: {sent}')
    print(f'  DE: {translate_attention(sent, encoder_model, decoder_model, tokenizer, max_encoder_len, max_decoder_len)}\n')

### 4.1. Quantitative evaluation

In [ ]:
# Evaluate on a larger sample
np.random.seed(315)
sample_indices = np.random.choice(len(pairs), size=min(200, len(pairs)), replace=False)

# Collect model predictions and ground truth
hypotheses = []  # Model translations
references = []  # Ground truth translations

print('Generating translations for BLEU evaluation...')

for i, idx in enumerate(sample_indices):

    en_text, de_ref = pairs[idx]
    de_hyp = translate_attention(en_text, encoder_model, decoder_model, tokenizer, max_encoder_len, max_decoder_len)
    
    hypotheses.append(de_hyp)
    references.append(de_ref)
    
    # Progress indicator
    if (i + 1) % 50 == 0:
        print(f'  Processed {i + 1}/{len(sample_indices)} samples')

# Compute corpus-level BLEU
bleu = BLEU()
result = bleu.corpus_score(hypotheses, [references])

print(f'\nBLEU score: {result.score:.2f}')
print(f'Breakdown: {result}')

# Qualitative analysis: inspect individual translations
print('\nSample predictions vs references:')

for i in range(5):

    print(f'  Source: {pairs[sample_indices[i]][0]}')
    print(f'  Reference: {references[i]}')
    print(f'  Hypothesis: {hypotheses[i]}\n')

## 5. Reflection questions

Answer these questions to deepen your understanding of transfer learning:

### Q1: Training efficiency
Compare the training time and convergence speed of fine-tuning (5 epochs) vs. training from scratch (15 epochs from notebook 02). Approximately how much faster is fine-tuning?

**Your answer:**
```
[TODO: Write your answer here]
```

### Q2: BLEU score comparison
Compare the final BLEU score after fine-tuning for 5 epochs vs. the BLEU score after training from scratch for 15 epochs (notebook 02). Is the fine-tuned model competitive?

**Your answer:**
```
[TODO: Write your answer here]
```

### Q3: What transfers?
Why do we reinitialize the embedding layers but keep the LSTM and attention weights? What would happen if we also reused the embedding weights?

**Your answer:**
```
[TODO: Write your answer here]
```

### Q4: Language similarity and transfer effectiveness
French and German are both European languages with similar word order (mostly SVO) and grammatical structures. Consider these factors and explain how they affect transfer learning:

a) **Word order:** English is SVO, German can be SOV in subordinate clauses. Japanese is strictly SOV. How would this affect transfer from English-French to English-Japanese?

b) **Script and morphology:** Arabic uses a different script, writes right-to-left, and has rich morphology. Would transfer learning from English-French to English-Arabic work as well? Why or why not?

c) **What's encoded in the LSTM weights:** Are the encoder's hidden states truly "language-agnostic" or are they tuned to the specific patterns of English-French? What evidence supports your answer?

**Your answer:**
```
[TODO: Write your answer here]
```

### Q5: Learning rate
We used the same learning rate for fine-tuning as for training from scratch. In practice, fine-tuning often uses a lower learning rate. Why might a lower learning rate be beneficial for fine-tuning?

**Your answer:**
```
[TODO: Write your answer here]
```

## 6. Extension challenges (optional)

Try these if you want to explore further:

1. **Compare from-scratch training:** Train an attention model for English-German from scratch (15 epochs) and compare final BLEU and training time with your fine-tuned model

2. **Try different learning rates:** Experiment with lower learning rates (0.0005, 0.0001) for fine-tuning and measure impact on convergence

3. **Freeze layers:** Try freezing the encoder and attention layers (make them non-trainable) and only training the decoder + embeddings. How does this affect performance?

4. **Different language pair:** Try fine-tuning for English-Spanish ('en-es') or English-Italian ('en-it'). Are results similar to English-German?

5. **Data efficiency:** Fine-tune with only 10,000 pairs instead of 100,000. How much does performance degrade? Compare with training from scratch on 10,000 pairs.